In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.lines import Line2D
import scienceplots
plt.style.use(['science', 'nature', 'grid'])

import numpy as np
from scipy.interpolate import interp1d
import pandas as pd

from LoadMoments import *
from LoadTSV import *

In [ ]:
# mpl.use('pgf')
mpl.rcParams.update({
    "pgf.texsystem": "pdflatex",
    "text.usetex": True,
    "font.family": "serif",
    "font.size": 10,        # Match your LaTeX document font size
    "axes.titlesize": 10,
    "axes.labelsize": 10,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8
})

In [ ]:
# Plotting
colors = ['blue', 'orange', 'green', 'red', 'purple', 'brown']
linestyles = ['-', '--', ':', '-.', (0, (3, 1, 1, 1)), (0, (5, 1))]

# Parameter
M = 4
closure = "ExtGram"
Kn = 1.0
source_string = "relaxation_source"
T_end = 0.3
base_tree_level = 10
polydeg = 1
rho_L = 7.0
v_L1 = 0.0
v_L2 = 0.0
v_L3 = 0.0
theta_L = 1.0
rho_R = 1.0
v_R1 = 0.0
v_R2 = 0.0
v_R3 = 0.0
theta_R = 1.0

# Solution files
n_angle_pairs = 4

marker = ['o', 's', '^', 'x', 'd']
angle_pairs = ["Max", "Arc", "Det"]#!, "Fibonacci"]

# Kn and relaxation for solution files
M_vec = [4, 6, 8]
source = 'relaxation_source'
# name = lambda anglepair, Kn, source_string: f"../out/1D3V/1D3V_angles_Edas_method/gram_solution_M{M}_closure{closure}_Kn{Kn}_source{source_string}_anglepair{anglepair}_cons.tsv"
name = lambda anglepair, Kn, source_string: f"../out/1D3V/1D3V_angles/gram_solution_M{M}_closure{closure}_Kn{Kn}_source{source_string}_anglepair{anglepair}_cons.tsv"

In [ ]:
# Semi-analytical solution -> Collisionless, 1D3V (slab geometry: v_y = v_z = 0)
trapz = getattr(np, "trapezoid", None) or np.trapz

class Maxwellian1D3V:
    def __init__(self, rho, v, theta):
        self.rho = rho
        self.v = np.asarray(v, dtype=float)      # (v_x, v_y, v_z)
        self.theta = theta

    def __call__(self, cx, cy, cz):
        """Full 3D Maxwellian, same normalisation as Maxwellian1D3V in ExtGram."""
        C2 = (cx - self.v[0])**2 + (cy - self.v[1])**2 + (cz - self.v[2])**2
        return self.rho / (2*np.pi*self.theta)**1.5 * np.exp(-C2 / (2*self.theta))

    def marginal(self, cx):
        """∫∫ f dcy dcz: the 1D Maxwellian in c_x that carries the density."""
        return self.rho / np.sqrt(2*np.pi*self.theta) * np.exp(-(cx - self.v[0])**2 / (2*self.theta))

    def transverse(self, c, axis):
        """Normalised 1D Maxwellian in c_y (axis=1) or c_z (axis=2)."""
        return 1 / np.sqrt(2*np.pi*self.theta) * np.exp(-(c - self.v[axis])**2 / (2*self.theta))

def f0(f_left, f_right, x, cx, cy, cz):
    x_arr, cx_arr, cy_arr, cz_arr = np.broadcast_arrays(np.asarray(x), np.asarray(cx), np.asarray(cy), np.asarray(cz))
    return np.where(x_arr < 0, f_left(cx_arr, cy_arr, cz_arr), f_right(cx_arr, cy_arr, cz_arr))

def f(f_left, f_right, x, cx, cy, cz, t):
    return f0(f_left, f_right, x - cx*t, cx, cy, cz)

def moment(f_left, f_right, i, j, k, x, t, Ncx, Nct):
    """
    U_(ijk)(x, t) = ∫ cx^i cy^j cz^k f(x, c, t) dc  for f = f0(x - cx t, c).
    The transport only involves cx, so per side the cy- and cz-integrals factor out and
    only the cx-integral depends on x (same trapz construction as in the 1D case).
    """
    x_arr, cx_arr = np.broadcast_arrays(np.asarray(x), Ncx[:, None])
    origin = x_arr - cx_arr*t                                  # where the particle started
    U = 0.0
    for side, weight in ((f_left, origin < 0), (f_right, origin >= 0)):
        Ix = trapz(weight * cx_arr**i * side.marginal(cx_arr), Ncx, axis=0)
        Iy = trapz(Nct**j * side.transverse(Nct, 1), Nct)
        Iz = trapz(Nct**k * side.transverse(Nct, 2), Nct)
        U = U + Ix * Iy * Iz
    return U

maxwellian_L = Maxwellian1D3V(rho_L, (v_L1, v_L2, v_L3), theta_L)
maxwellian_R = Maxwellian1D3V(rho_R, (v_R1, v_R2, v_R3), theta_R)

x_ex = np.linspace(-2, 2, 2001)
Ncx = np.linspace(-10, 10, 4001)        # streaming direction: needs the fine grid
Nct = np.linspace(-10, 10, 2001)        # transverse directions: 1D Gaussian integrals only
T = 0.3


U = lambda i, j, k: moment(maxwellian_L, maxwellian_R, i, j, k, x_ex, T, Ncx, Nct)
rho_ex = U(0, 0, 0)
v_ex = U(1, 0, 0) / rho_ex
theta_ex = (U(2, 0, 0) - rho_ex*v_ex**2 + U(0, 2, 0) + U(0, 0, 2)) / (3*rho_ex)   # trace of the pressure tensor / (3 rho)
p_ex = rho_ex * theta_ex

In [ ]:
for M in [4, 6, 8]:
    fig, axs = plt.subplots(1, 3, sharex=True, sharey=True, figsize=(6, 3))

    twx_dict = {}
    first_twx = None
    kn_titles = [r'Kn=0.1', r'Kn=1.0', r'Kn$\to \infty$']

    # 2. Setup Axes and clean up labels
    for ax in axs.flatten():
        ax.set_xlim(-1, 1)
        
        twx = ax.twinx()
        twx.grid(False)
        
        if first_twx is None:
            first_twx = twx
        else:
            twx.sharey(first_twx)
            
        twx_dict[ax] = twx

    # Apply outer labels only (prevents overlapping messes)
    for ax in axs:
        ax.set_ylabel(r'$\rho, p$')
    for ax in axs:
        ax.set_xlabel('x')
    twx.set_ylabel('$v$')

    # Dummy plots for legend (colors)
    for color, label in zip(colors, [r'$\rho$', r'$v$', r'$p$']):
        axs[0].plot([], [], color=color, label=label)

    # 3. Data Loop with Row and Column Titles
    for row, angle_pair in enumerate(angle_pairs):
        
        # Set Row Titles on the far right
        twx_right = twx_dict[axs[2]]

        for column, (Kn, source) in enumerate([(0.1, 'relaxation_source'), (1.0, 'relaxation_source'), (10.0, 'zero_source')]):
            ax = axs[column]
            twx = twx_dict[ax] 

            if source == 'zero_source':
                ax.plot(x_ex, rho_ex, color='gray', linestyle='-', lw=1, alpha=0.5)
                twx.plot(x_ex, v_ex, color='gray', linestyle='-', lw=1, alpha=0.5)
                ax.plot(x_ex, p_ex, color='gray', linestyle='-', lw=1, alpha=0.5)
            
            # Set Column Titles on the top row only
            if row == 0:
                ax.set_title(kn_titles[column])

            ## Reduced 1D-3V slab (Assuming your data reading functions work)
            solution_file = name(angle_pair, Kn, source)
            data = read_solution_file(solution_file)
            
            times = np.array([data[ts]["t"] for ts in sorted(data.keys())])
            timesteps = np.array(sorted(data.keys()))

            if len(timesteps) != 11: # Solution diverged before
                print(f"Warning: Expected 11 timesteps for M={M}, Kn={Kn}, source={source}, angle_pair={angle_pair}, but got {len(timesteps)}. Skipping this case.")
                continue  # Skip this iteration if the number of timesteps is not as expected
            
            time_query = 21 
            sol = interpolate_solution(data, time_query, times, timesteps)
            x, U = sol["x"], sol["u"]

            rho = U[:, 0]
            v = U[:, 1] / U[:, 0]
            theta = 1 / (3 * rho) * (U[:, 2] + 2 * U[:, 3] - rho * v**2) 
            p = rho * theta

            ax.plot(x, rho, color=colors[0], linestyle=linestyles[row])
            twx.plot(x, v, color=colors[1], linestyle=linestyles[row])
            ax.plot(x, p, color=colors[2], linestyle=linestyles[row])

            # Dummy plot for legend
            if column == 0:
                ax.plot([], [], color='black', linestyle=linestyles[row], lw=1, label=angle_pair)

    # Strip inner labels for standard axes
    for ax in axs.flat:
        ax.label_outer()
    axs[0].plot([], [], color='gray', linestyle='-', lw=1, alpha=0.5, label='Reference Solution')

    fig.legend(loc='lower center', ncol=7)
    fig.tight_layout(rect=(0, 0.05, 1, 1))
    fig.savefig(f'../out/Figures/1D3V/slab/AngleComparison_M{M}.pgf')
    fig.show()

In [ ]:
# Plot the highest two moments
for M in [4, 6, 8]:
    fig, axs = plt.subplots(1, 3, sharex=True, sharey=True, figsize=(6, 3))

    twx_dict = {}
    first_twx = None
    kn_titles = [r'Kn=0.1', r'Kn=1.0', r'Kn$\to \infty$']

    # 2. Setup Axes and clean up labels
    for ax in axs.flatten():
        ax.set_xlim(-1, 1)
        
        twx = ax.twinx()
        twx.grid(False)
        
        if first_twx is None:
            first_twx = twx
        else:
            twx.sharey(first_twx)
            
        twx_dict[ax] = twx

    # Apply outer labels only (prevents overlapping messes)
    for ax in axs:
        ax.set_ylabel(r'$\rho, p$')
    for ax in axs:
        ax.set_xlabel('x')
    twx.set_ylabel('$v$')

    # Dummy plots for legend (colors)
    for color, label in zip(colors, [r'Highest moment', r'Second highest moment']):
        axs[0].plot([], [], color=color, label=label)

    # 3. Data Loop with Row and Column Titles
    for row, angle_pair in enumerate(angle_pairs):
        
        # Set Row Titles on the far right
        twx_right = twx_dict[axs[2]]

        for column, (Kn, source) in enumerate([(0.1, 'relaxation_source'), (1.0, 'relaxation_source'), (10.0, 'zero_source')]):
            ax = axs[column]
            twx = twx_dict[ax] 
            
            # Set Column Titles on the top row only
            if row == 0:
                ax.set_title(kn_titles[column])

            ## Reduced 1D-3V slab (Assuming your data reading functions work)
            solution_file = name(angle_pair, Kn, source)
            data = read_solution_file(solution_file)
            
            times = np.array([data[ts]["t"] for ts in sorted(data.keys())])
            timesteps = np.array(sorted(data.keys()))

            if len(timesteps) != 11: # Solution diverged before
                print(f"Warning: Expected 11 timesteps for M={M}, Kn={Kn}, source={source}, angle_pair={angle_pair}, but got {len(timesteps)}. Skipping this case.")
                continue  # Skip this iteration if the number of timesteps is not as expected
            
            time_query = 21 
            sol = interpolate_solution(data, time_query, times, timesteps)
            x, U = sol["x"], sol["u"]

            ax.plot(x, U[:, -1], color=colors[0], linestyle=linestyles[row])
            twx.plot(x, U[:, -2], color=colors[1], linestyle=linestyles[row])

            # Dummy plot for legend
            if column == 0:
                ax.plot([], [], color='black', linestyle=linestyles[row], lw=1, label=angle_pair)

    # Strip inner labels for standard axes
    for ax in axs.flat:
        ax.label_outer()
    axs[0].plot([], [], color='gray', linestyle='-', lw=1, alpha=0.5, label='Reference Solution')

    fig.legend(loc='lower center', ncol=7)
    fig.tight_layout(rect=(0, 0.05, 1, 1))
    # fig.savefig(f'../out/Figures/1D3V/slab/AngleComparison_M{M}.pgf')
    fig.show()

In [ ]:
# Model Convergence for Max Angle
for angle_pair in angle_pairs:
    fig, axs = plt.subplots(1, 3, sharex=True, sharey=True, figsize=(6, 3))

    twx_dict = {}
    first_twx = None
    kn_titles = [r'Kn=0.1', r'Kn=1.0', r'Kn$\to \infty$']

    # 2. Setup Axes and clean up labels
    for ax in axs.flatten():
        ax.set_xlim(-1, 1)
        
        twx = ax.twinx()
        twx.grid(False)
        
        if first_twx is None:
            first_twx = twx
        else:
            twx.sharey(first_twx)
            
        twx_dict[ax] = twx

    # Apply outer labels only (prevents overlapping messes)
    for ax in axs:
        ax.set_ylabel(r'$\rho, p$')
    for ax in axs:
        ax.set_xlabel('x')
    twx.set_ylabel('$v$')

    # Dummy plots for legend (colors)
    for color, label in zip(colors, [r'$\rho$', r'$v$', r'$p$']):
        axs[0].plot([], [], color=color, label=label)

    for i, M in enumerate(M_vec):
        for column, (Kn, source) in enumerate([(0.1, 'relaxation_source'), (1.0, 'relaxation_source'), (10.0, 'zero_source')]):
            ax = axs[column]
            twx = twx_dict[ax] 

            if source == 'zero_source':
                ax.plot(x_ex, rho_ex, color='gray', linestyle='-', lw=1, alpha=0.5)
                twx.plot(x_ex, v_ex, color='gray', linestyle='-', lw=1, alpha=0.5)
                ax.plot(x_ex, p_ex, color='gray', linestyle='-', lw=1, alpha=0.5)
            
            # Set Column Titles on the top row only
            if row == 0:
                ax.set_title(kn_titles[column])

            ## Reduced 1D-3V slab (Assuming your data reading functions work)
            solution_file = name(angle_pair, Kn, source)
            data = read_solution_file(solution_file)
            
            times = np.array([data[ts]["t"] for ts in sorted(data.keys())])
            timesteps = np.array(sorted(data.keys()))

            if len(timesteps) != 11: # Solution diverged before
                print(f"Warning: Expected 11 timesteps for M={M}, Kn={Kn}, source={source}, angle_pair={angle_pair}, but got {len(timesteps)}. Skipping this case.")
                continue  # Skip this iteration if the number of timesteps is not as expected
            
            time_query = 21 
            sol = interpolate_solution(data, time_query, times, timesteps)
            x, U = sol["x"], sol["u"]

            rho = U[:, 0]
            v = U[:, 1] / U[:, 0]
            theta = 1 / (3 * rho) * (U[:, 2] + 2 * U[:, 3] - rho * v**2) 
            p = rho * theta

            ax.plot(x, rho, color=colors[0], linestyle=linestyles[i])
            twx.plot(x, v, color=colors[1], linestyle=linestyles[i])
            ax.plot(x, p, color=colors[2], linestyle=linestyles[i])

            # Dummy plot for legend
            if column == 0:
                ax.plot([], [], color='black', linestyle=linestyles[i], lw=1, label=f'M={M}')

    # Strip inner labels for standard axes
    for ax in axs.flat:
        ax.label_outer()
    axs[0].plot([], [], color='gray', linestyle='-', lw=1, alpha=0.5, label='Reference Solution')

    fig.legend(loc='lower center', ncol=7)
    fig.tight_layout(rect=(0, 0.05, 1, 1))
    fig.savefig(f'../out/Figures/1D3V/slab/ModelConvergence_closure{closure}_angle{angle_pair}.pgf')
    fig.show()

In [ ]:
fig, axs = plt.subplots(3, 3, sharex=True, sharey=True, figsize=(6, 8))

twx_dict = {}
first_twx = None
kn_titles = [r'Kn=0.1', r'Kn=1.0', r'Kn$\to \infty$']

# 2. Setup Axes and clean up labels
for ax in axs.flatten():
    ax.set_xlim(-1, 1)
    
    twx = ax.twinx()
    twx.grid(False)
    
    if first_twx is None:
        first_twx = twx
    else:
        twx.sharey(first_twx)
        
    twx_dict[ax] = twx

# Apply outer labels only (prevents overlapping messes)
for ax in axs[:, 0]:
    ax.set_ylabel(r'$\rho, p$')
for ax in axs[-1, :]:
    ax.set_xlabel('x')
for ax in axs[:, -1]:
    twx_dict[ax].set_ylabel('$v$')

# Dummy plots for legend (colors)
for color, label in zip(colors, [r'$\rho$', r'$v$', r'$p$']):
    axs[0, 0].plot([], [], color=color, label=label)

# 3. Data Loop with Row and Column Titles
for row, angle_pair in enumerate(angle_pairs):
    
    # Set Row Titles on the far right
    twx_right = twx_dict[axs[row, 2]]
    twx_right.annotate(f'Angles: {angle_pair}', xy=(1, 0.5), xycoords=twx_right.yaxis.label,
                       xytext=(4, 0), textcoords='offset points',
                       rotation=-90, va='center', ha='left', fontsize=10)

    for i, M in enumerate(M_vec):
        for column, (Kn, source) in enumerate([(0.1, 'relaxation_source'), (1.0, 'relaxation_source'), (10.0, 'zero_source')]):
            ax = axs[row, column]
            twx = twx_dict[ax] 
            
            # Set Column Titles on the top row only
            if row == 0:
                ax.set_title(kn_titles[column])

            ## Reduced 1D-3V slab (Assuming your data reading functions work)
            solution_file = name(angle_pair, Kn, source)
            data = read_solution_file(solution_file)
            
            times = np.array([data[ts]["t"] for ts in sorted(data.keys())])
            timesteps = np.array(sorted(data.keys()))

            if len(timesteps) != 11: # Solution diverged before
                print(f"Warning: Expected 11 timesteps for M={M}, Kn={Kn}, source={source}, angle_pair={angle_pair}, but got {len(timesteps)}. Skipping this case.")
                continue  # Skip this iteration if the number of timesteps is not as expected
            
            time_query = 21 
            sol = interpolate_solution(data, time_query, times, timesteps)
            x, U = sol["x"], sol["u"]

            rho = U[:, 0]
            v = U[:, 1] / U[:, 0]
            theta = 1 / (3 * rho) * (U[:, 2] + 2 * U[:, 3] - rho * v**2) 
            p = rho * theta

            ax.plot(x, rho, color=colors[0], linestyle=linestyles[i])
            twx.plot(x, v, color=colors[1], linestyle=linestyles[i])
            ax.plot(x, p, color=colors[2], linestyle=linestyles[i])

            # Dummy plot for legend
            if row == 0 and column == 0:
                ax.plot([], [], color='black', linestyle=linestyles[i], lw=1, label=f'M={M}')

            if source == 'zero_source':
                ax.plot(x_ex, rho_ex, color='gray', linestyle='-', lw=1, alpha=0.5)
                twx.plot(x_ex, v_ex, color='gray', linestyle='-', lw=1, alpha=0.5)
                ax.plot(x_ex, p_ex, color='gray', linestyle='-', lw=1, alpha=0.5)

# Strip inner labels for standard axes
for ax in axs.flat:
    ax.label_outer()
axs[0, 0].plot([], [], color='gray', linestyle='-', lw=1, alpha=0.5, label='Reference Solution')

fig.legend(loc='lower center', ncol=7)
fig.tight_layout(rect=(0, 0.05, 1, 1))
fig.savefig(f'../out/Figures/1D3V/slab/ModelConvergence_closure{closure}.pgf')
fig.show()